In [38]:
# Step 5 - Sample Diagnostic.

# Verification: what's actually in the saved sample?
verify = pd.read_csv(sample_path, nrows=5)
print(f"Sample column count: {len(verify.columns)}")
print(f"Sample columns:")
for col in verify.columns:
    print(f"  - {col}")

Sample column count: 28
Sample columns:
  - id
  - loan_amnt
  - funded_amnt
  - term
  - int_rate
  - installment
  - grade
  - sub_grade
  - emp_length
  - home_ownership
  - annual_inc
  - verification_status
  - issue_d
  - loan_status
  - purpose
  - addr_state
  - dti
  - delinq_2yrs
  - earliest_cr_line
  - fico_range_low
  - fico_range_high
  - inq_last_6mths
  - mths_since_last_delinq
  - open_acc
  - revol_bal
  - revol_util
  - total_acc
  - application_type


In [37]:
# Step 4 - Stratified Sample.

# Stratified sample: preserve default rate proportions
sample_size = 200_000

# Compute proportional sample sizes per class
paid_ratio = (df_terminal["loan_status"] == "Fully Paid").sum() / len(df_terminal)
charged_ratio = (df_terminal["loan_status"] == "Charged Off").sum() / len(df_terminal)

paid_sample_size = int(sample_size * paid_ratio)
charged_sample_size = sample_size - paid_sample_size

# Sample from each class with fixed random_state for reproducibility
paid_sample = df_terminal[df_terminal["loan_status"] == "Fully Paid"].sample(
    n=paid_sample_size, random_state=42
)
charged_sample = df_terminal[df_terminal["loan_status"] == "Charged Off"].sample(
    n=charged_sample_size, random_state=42
)

# Combine into working sample
df_sample = pd.concat([paid_sample, charged_sample], ignore_index=True)

# Shuffle so it's not sorted by class
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sample shape: {df_sample.shape}")
print(f"Sample class balance:")
print(df_sample["loan_status"].value_counts(normalize=True).round(4))
print()
print(f"Sample memory: {df_sample.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# Save sample for reuse across notebooks
sample_path = Path().resolve().parent / "data" / "sample_200k_stratified.csv"
df_sample.to_csv(sample_path, index=False)
print(f"Sample saved: {sample_path}")
print(f"File size: {sample_path.stat().st_size / (1024**2):.2f} MB")

Sample shape: (200000, 28)
Sample class balance:
loan_status
Fully Paid     0.8004
Charged Off    0.1996
Name: proportion, dtype: float64

Sample memory: 159.20 MB
Sample saved: C:\Users\Mark PC\dev\data-analyst-learning-journal\capstone-lending-club\data\sample_200k_stratified.csv
File size: 38.11 MB


In [36]:
# Step 3 — Filter to Terminal Loans.
# Assumes: df already loaded in Step 2 with 28 selected columns.

# Verify state before proceeding
print(f"df shape at start of Step 3: {df.shape}")
assert df.shape[1] == 28, f"Expected 28 columns, got {df.shape[1]}"

# Loan status distribution in full data
status_counts = df["loan_status"].value_counts()
print("Loan status distribution (full 2.26M):")
print(status_counts)
print()

# Total terminal loans (defined as Fully Paid + Charged Off)
terminal_mask = df["loan_status"].isin(["Fully Paid", "Charged Off"])
terminal_count = terminal_mask.sum()
print(f"Terminal loans: {terminal_count:,}")
print(f"Current + other non-terminal: {(~terminal_mask).sum():,}")

# Default rate on terminal loans
charged_off_count = (df["loan_status"] == "Charged Off").sum()
default_rate = charged_off_count / terminal_count * 100
print(f"Default rate on terminal loans: {default_rate:.2f}%")

# Filter to terminal loans only for capstone analysis
df_terminal = df[terminal_mask].copy()
print(f"Terminal-loan DataFrame shape: {df_terminal.shape}")
print(f"Memory: {df_terminal.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
assert df_terminal.shape[1] == 28, f"Expected 28 columns, got {df_terminal.shape[1]}"

# Free up the full DataFrame from memory (we won't need it until validation)
del df
import gc
gc.collect()

print("\n✅ Step 3 complete — df_terminal ready with 28 columns")

df shape at start of Step 3: (2260701, 28)
Loan status distribution (full 2.26M):
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

Terminal loans: 1,345,310
Current + other non-terminal: 915,391
Default rate on terminal loans: 19.96%
Terminal-loan DataFrame shape: (1345310, 28)
Memory: 1081.17 MB

✅ Step 3 complete — df_terminal ready with 28 columns


In [35]:
# Step 2 — Load with column selection.
# Load only the 28 features defined in Step 1.

df = pd.read_csv(csv_path, usecols=FEATURES)
print(f"Full loaded shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

assert df.shape[1] == 28, f"Expected 28 columns, got {df.shape[1]}"
print("\n✅ Step 2 complete — df loaded with 28 columns")

C:\Users\Mark PC\AppData\Local\Temp\ipykernel_11776\413073035.py:4: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, usecols=FEATURES)


Full loaded shape: (2260701, 28)
Memory: 1796.99 MB

✅ Step 2 complete — df loaded with 28 columns


In [30]:
# Step 1 — Load selected features from full file 

import pandas as pd
from pathlib import Path

csv_path = Path("../data/accepted_2007_to_2018Q4.csv")

# Feature list from CAPSTONE_SCOPING.md
CORE_LOAN_FIELDS = [
    "id", "loan_amnt", "funded_amnt", "term", "int_rate", 
    "installment", "grade", "sub_grade", "loan_status", 
    "purpose", "issue_d", "application_type", "addr_state"
]

BORROWER_CHARACTERISTICS = [
    "annual_inc", "verification_status", "emp_length", 
    "home_ownership", "dti", "fico_range_low", "fico_range_high", 
    "earliest_cr_line", "open_acc", "total_acc", "revol_bal", 
    "revol_util", "inq_last_6mths", "delinq_2yrs", 
    "mths_since_last_delinq"
]

FEATURES = CORE_LOAN_FIELDS + BORROWER_CHARACTERISTICS
print(f"Selected features: {len(FEATURES)}")

Selected features: 28


# Capstone: Lending Club Loan Default Analysis

**Notebook 02:** 200K stratified sample creation + first-pass exploration.

**Purpose:** Create the development sample that all subsequent analytical 
notebooks will use. Establish first-pass understanding of the ~28 features 
selected during scoping.

**Not in this notebook:** analytical questions, findings, charts beyond 
basic descriptive statistics.

**Reference:** See `../CAPSTONE_SCOPING.md` for full analytical scope.